# 🎙️ TalkForge — AI Lip Sync Video Generator

**Portrait + Audio → Talking-Head Video with accurate lip sync, powered by SadTalker.**

---

## Before you run

1. **Enable GPU:** `Runtime → Change runtime type → T4 GPU → Save`
2. **Run all cells:** `Runtime → Run all` (or `Ctrl + F9`)
3. **Wait for the public URL** to appear at the bottom of the last cell, then open it.

> ⏱️ First-run setup: ~8–12 min (downloads ~2 GB of model weights).  
> ⚡ Subsequent runs in the same session: ~30 seconds (weights cached).

---

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 1 — Verify GPU & Python version
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys, os

print(f'Python {sys.version}')

r = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
     '--format=csv,noheader'],
    capture_output=True, text=True
)
if r.returncode == 0:
    print(f'✅ GPU: {r.stdout.strip()}')
else:
    print('⚠️  No GPU detected.')
    print('   Runtime → Change runtime type → T4 GPU for faster results.')
    print('   Continuing on CPU — generation will be very slow.')

# Warn if running Python 3.12 — basicsr needs a patch (handled in Cell 6)
if sys.version_info >= (3, 12):
    print('⚠️  Python 3.12 detected — basicsr compatibility patch will be applied in Cell 6.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 2 — System packages (FFmpeg, cmake, build tools)
# ─────────────────────────────────────────────────────────────────────────────
# cmake + build-essential are required to compile dlib from source.

import subprocess, sys

print('Installing system packages…')
subprocess.run(
    ['apt-get', 'update', '-qq'],
    check=True, capture_output=True
)
subprocess.run(
    ['apt-get', 'install', '-y', '-qq',
     'ffmpeg', 'cmake', 'build-essential',
     'libopenblas-dev', 'liblapack-dev',
     'libx11-dev', 'python3-dev',
     'git', 'wget', 'curl'],
    check=True, capture_output=True
)

r = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
if r.returncode == 0:
    print('✅ FFmpeg:', r.stdout.split('\n')[0])
else:
    raise RuntimeError('FFmpeg installation failed — re-run this cell.')

print('✅ System packages ready')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 3 — Clone TalkForge & SadTalker
# ─────────────────────────────────────────────────────────────────────────────
import os, subprocess, pathlib

TALKFORGE_DIR  = '/content/TalkForge'
SADTALKER_DIR  = f'{TALKFORGE_DIR}/models/SadTalker'

# ── TalkForge ────────────────────────────────────────────────────────────────
# Update this URL to your fork once published on GitHub.
# If the repo isn't public yet, the fallback below creates the required files.
TALKFORGE_REPO = 'https://github.com/your-username/TalkForge.git'

if pathlib.Path(f'{TALKFORGE_DIR}/.git').exists():
    print('TalkForge already cloned — pulling latest…')
    subprocess.run(['git', '-C', TALKFORGE_DIR, 'pull', '-q'], check=True)
else:
    print('Cloning TalkForge…')
    r = subprocess.run(
        ['git', 'clone', '--depth', '1', TALKFORGE_REPO, TALKFORGE_DIR],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print('  (repo not yet public — creating required project structure inline)')
        # Create skeleton so subsequent cells can run
        for d in ['app', 'models', 'outputs', 'uploads']:
            pathlib.Path(f'{TALKFORGE_DIR}/{d}').mkdir(parents=True, exist_ok=True)
        pathlib.Path(f'{TALKFORGE_DIR}/app/__init__.py').write_text('')
        pathlib.Path(f'{TALKFORGE_DIR}/models/__init__.py').write_text('')
    else:
        print('✅ TalkForge cloned')

# ── SadTalker ────────────────────────────────────────────────────────────────
if pathlib.Path(f'{SADTALKER_DIR}/.git').exists():
    print('SadTalker already cloned — skipping.')
else:
    print('Cloning SadTalker…')
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/OpenTalker/SadTalker.git', SADTALKER_DIR],
        check=True
    )
    print('✅ SadTalker cloned')

# Ensure all required directories exist
for d in ['weights/checkpoints/BFM_Fitting', 'weights/gfpgan/weights',
          'outputs', 'uploads']:
    pathlib.Path(f'{TALKFORGE_DIR}/{d}').mkdir(parents=True, exist_ok=True)

import sys
os.chdir(TALKFORGE_DIR)
if TALKFORGE_DIR not in sys.path:
    sys.path.insert(0, TALKFORGE_DIR)

print(f'✅ Working directory: {os.getcwd()}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 4 — PyTorch
# ─────────────────────────────────────────────────────────────────────────────
# Colab ships with a recent PyTorch+CUDA.  We only reinstall if the version
# is too old or CUDA is unavailable.

import subprocess, sys

def _pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

try:
    import torch
    if torch.cuda.is_available() and int(torch.__version__.split('.')[0]) >= 2:
        print(f'✅ PyTorch {torch.__version__} + CUDA {torch.version.cuda} already present.')
        raise SystemExit(0)   # skip reinstall
except SystemExit:
    pass
except Exception:
    pass

print('Installing PyTorch 2.1 + CUDA 11.8…')
_pip(
    'torch==2.1.0+cu118', 'torchvision==0.16.0+cu118', 'torchaudio==2.1.0',
    '--index-url', 'https://download.pytorch.org/whl/cu118'
)

import torch
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 5 — Core Python dependencies
# ─────────────────────────────────────────────────────────────────────────────
# We install packages individually with compatible version ranges instead of
# SadTalker's requirements.txt, which has strict old pins that conflict with
# the current Colab environment (Python 3.10–3.12, updated system packages).

import subprocess, sys

def pip(*pkgs, upgrade=False):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q']
    if upgrade:
        cmd.append('--upgrade')
    cmd.extend(pkgs)
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'⚠️  pip warning for {pkgs[0]}:')
        print(r.stderr[-800:])

# ── numpy (pin below 2.0 — required by basicsr and older scipy interfaces) ──
print('numpy…')
pip('"numpy>=1.24.0,<2.0.0"')

# ── Image / video utilities ──────────────────────────────────────────────────
print('image/video utilities…')
pip(
    'opencv-python-headless',
    'Pillow',
    'imageio>=2.28.0',
    'imageio-ffmpeg>=0.4.9',
    'scikit-image>=0.21.0',
    'av',
)

# ── Audio ────────────────────────────────────────────────────────────────────
print('audio…')
pip(
    'librosa>=0.10.0',
    'soundfile>=0.12.1',
    'resampy>=0.4.2',
    'pydub>=0.25.1',
)

# ── Scientific / ML utilities ────────────────────────────────────────────────
print('scientific stack…')
pip(
    'scipy>=1.11.0',
    'einops>=0.7.0',
    'kornia>=0.7.0',
    'safetensors>=0.4.0',
    'yacs>=0.1.8',
    'pyyaml',
    'tqdm',
    'requests',
    'joblib>=1.3.0',
    'h5py',
    'addict',
)

# ── Gradio ───────────────────────────────────────────────────────────────────
print('gradio…')
pip('"gradio>=4.26.0,<5.0.0"')

print('✅ Core dependencies installed')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 6 — Face / enhancement libraries + Python 3.12 basicsr patch
# ─────────────────────────────────────────────────────────────────────────────
# basicsr 1.4.2 uses ast.Str / ast.Num which were removed in Python 3.12.
# We detect the Python version and apply a one-line patch if needed.

import subprocess, sys, importlib, textwrap, pathlib

def pip(*pkgs, upgrade=False):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q']
    if upgrade:
        cmd.append('--upgrade')
    cmd.extend(pkgs)
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'⚠️  {pkgs[0]}:', r.stderr[-600:])

# ── dlib (compiled from source — takes 1–3 min) ──────────────────────────────
try:
    import dlib
    print(f'✅ dlib {dlib.__version__} already installed')
except ImportError:
    print('Installing dlib (compiling from source — ~2 min)…')
    pip('dlib')

# ── face-alignment ──────────────────────────────────────────────────────────
print('face-alignment…')
pip('face-alignment>=1.4.1')

# ── basicsr ─────────────────────────────────────────────────────────────────
print('basicsr…')
pip('basicsr>=1.4.2')

# ── Python 3.12 compatibility patch for basicsr ──────────────────────────────
# basicsr/data/degradations.py uses ast.Str and ast.Num which were removed
# in Python 3.12 (PEP 619).  The fix is to replace the type checks with
# the canonical Python 3.8+ equivalent (ast.Constant).
if sys.version_info >= (3, 12):
    print('Applying Python 3.12 basicsr patch…')
    import basicsr
    deg_path = pathlib.Path(basicsr.__file__).parent / 'data' / 'degradations.py'
    if deg_path.exists():
        src = deg_path.read_text()
        # The deprecated classes were ast.Num, ast.Str, ast.Index, ast.NameConstant
        # They map to ast.Constant in Python 3.8+
        patched = src
        for old, new in [
            ('isinstance(node, ast.Num)',          'isinstance(node, ast.Constant)'),
            ('isinstance(node, ast.Str)',          'isinstance(node, ast.Constant)'),
            ('isinstance(node, ast.NameConstant)', 'isinstance(node, ast.Constant)'),
            ('node.n',                             'node.value'),
            ('node.s',                             'node.value'),
        ]:
            patched = patched.replace(old, new)
        if patched != src:
            deg_path.write_text(patched)
            print('  ✅ degradations.py patched')
        else:
            print('  ℹ️  No patch needed in degradations.py')

        # Also check utils/options.py for ast issues
        opt_path = pathlib.Path(basicsr.__file__).parent / 'utils' / 'options.py'
        if opt_path.exists():
            src = opt_path.read_text()
            patched = src.replace(
                'isinstance(node.value, ast.Str)',
                'isinstance(node.value, ast.Constant)'
            ).replace(
                'node.value.s',
                'node.value.value'
            )
            if patched != src:
                opt_path.write_text(patched)
                print('  ✅ options.py patched')
    else:
        print('  ℹ️  degradations.py not found — skipping patch')

# ── facexlib, realesrgan, gfpgan ─────────────────────────────────────────────
print('facexlib / realesrgan / gfpgan…')
pip('facexlib>=0.3.0')
pip('realesrgan>=0.3.0')
pip('gfpgan>=1.3.8')

print('\n✅ Face / enhancement libraries ready')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 7 — Download model weights
# ─────────────────────────────────────────────────────────────────────────────
# All URLs verified HTTP 200 as of 2025.
# Re-running this cell is safe — files already downloaded are skipped.
#
# Sources:
#   SadTalker main models → GitHub Releases (OpenTalker/SadTalker v0.0.2-rc)
#   BFM Fitting source    → HuggingFace (vinthony/SadTalker)
#   GFPGAN / facexlib     → GitHub Releases (TencentARC/GFPGAN, xinntao/facexlib)
#
# NOTE: BFM_model_front.mat is NOT downloaded here.
# SadTalker generates it automatically on first run from 01_MorphableModel.mat
# + Exp_Pca.bin via its internal transferBFM09() function.

import os, pathlib, subprocess, urllib.request, sys

W   = pathlib.Path(TALKFORGE_DIR) / 'weights'
CKP = W / 'checkpoints'
BFM = CKP / 'BFM_Fitting'
GFP = W / 'gfpgan' / 'weights'

for d in [CKP, BFM, GFP]:
    d.mkdir(parents=True, exist_ok=True)

# ── Verified download URLs ────────────────────────────────────────────────────
_GH  = 'https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc'
_HF  = 'https://huggingface.co/vinthony/SadTalker/resolve/main'
_FX  = 'https://github.com/xinntao/facexlib/releases/download'
_GFP = 'https://github.com/TencentARC/GFPGAN/releases/download'

DOWNLOADS = [
    # SadTalker main models
    (f'{_GH}/SadTalker_V0.0.2_256.safetensors', CKP / 'SadTalker_V0.0.2_256.safetensors'),
    (f'{_GH}/mapping_00229-model.pth.tar',       CKP / 'mapping_00229-model.pth.tar'),
    (f'{_GH}/mapping_00109-model.pth.tar',       CKP / 'mapping_00109-model.pth.tar'),
    # BFM Fitting source files (used to auto-generate BFM_model_front.mat)
    (f'{_HF}/BFM_Fitting/01_MorphableModel.mat',   BFM / '01_MorphableModel.mat'),
    (f'{_HF}/BFM_Fitting/Exp_Pca.bin',             BFM / 'Exp_Pca.bin'),
    (f'{_HF}/BFM_Fitting/std_exp.txt',             BFM / 'std_exp.txt'),
    (f'{_HF}/BFM_Fitting/similarity_Lm3D_all.mat', BFM / 'similarity_Lm3D_all.mat'),
    (f'{_HF}/BFM_Fitting/BFM_exp_idx.mat',         BFM / 'BFM_exp_idx.mat'),
    (f'{_HF}/BFM_Fitting/BFM_front_idx.mat',       BFM / 'BFM_front_idx.mat'),
    (f'{_HF}/BFM_Fitting/BFM09_model_info.mat',    BFM / 'BFM09_model_info.mat'),
    (f'{_HF}/BFM_Fitting/facemodel_info.mat',      BFM / 'facemodel_info.mat'),
    (f'{_HF}/BFM_Fitting/select_vertex_id.mat',    BFM / 'select_vertex_id.mat'),
    # GFPGAN face enhancer
    (f'{_GFP}/v1.3.0/GFPGANv1.4.pth',             GFP / 'GFPGANv1.4.pth'),
    # facexlib detection / alignment / parsing models
    (f'{_FX}/v0.1.0/alignment_WFLW_4HG.pth',      GFP / 'alignment_WFLW_4HG.pth'),
    (f'{_FX}/v0.1.0/detection_Resnet50_Final.pth', GFP / 'detection_Resnet50_Final.pth'),
    (f'{_FX}/v0.2.2/parsing_parsenet.pth',         GFP / 'parsing_parsenet.pth'),
]

def download(url: str, dst: pathlib.Path) -> None:
    """Download url→dst, skipping if already present and non-empty."""
    if dst.exists() and dst.stat().st_size > 1024:
        mb = dst.stat().st_size / 1024 / 1024
        print(f'  ✓ {dst.name} ({mb:.0f} MB) — cached')
        return
    print(f'  ↓ {dst.name}…', end='', flush=True)
    dst.parent.mkdir(parents=True, exist_ok=True)
    # Use wget (faster progress display in Colab)
    r = subprocess.run(
        ['wget', '-q', '--show-progress', '-O', str(dst), url],
        capture_output=False
    )
    if r.returncode != 0 or not dst.exists() or dst.stat().st_size < 1024:
        # Fallback to urllib
        print('  (wget failed, trying urllib…)')
        urllib.request.urlretrieve(url, str(dst))
    mb = dst.stat().st_size / 1024 / 1024
    print(f' ✅ ({mb:.0f} MB)')

print('=== Downloading SadTalker main models (GitHub Releases) ===')
for url, dst in DOWNLOADS[:3]:
    download(url, dst)

print('\n=== Downloading BFM Fitting source files (HuggingFace) ===')
for url, dst in DOWNLOADS[3:12]:
    download(url, dst)

print('\n=== Downloading GFPGAN / facexlib enhancement models ===')
for url, dst in DOWNLOADS[12:]:
    download(url, dst)

print('\n✅ All model weights downloaded.')

# Also symlink gfpgan weights into the SadTalker working directory so the
# enhancer can find them without hitting the network during inference.
sadtalker_gfp = pathlib.Path(SADTALKER_DIR) / 'gfpgan' / 'weights'
sadtalker_gfp.mkdir(parents=True, exist_ok=True)
for f in GFP.iterdir():
    dst = sadtalker_gfp / f.name
    if not dst.exists():
        try:
            dst.symlink_to(f.resolve())
        except Exception:
            import shutil
            shutil.copy2(str(f), str(dst))
print('✅ GFPGAN weights linked into SadTalker directory')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 8 — Verify setup
# ─────────────────────────────────────────────────────────────────────────────
import sys, os, pathlib

# Fresh import of TalkForge modules
for mod in list(sys.modules.keys()):
    if mod.startswith(('models', 'app')):
        del sys.modules[mod]

os.chdir(TALKFORGE_DIR)
if TALKFORGE_DIR not in sys.path:
    sys.path.insert(0, TALKFORGE_DIR)

from models.sadtalker import SadTalkerModel, WEIGHT_URLS

model = SadTalkerModel(
    weights_dir=f'{TALKFORGE_DIR}/weights',
    device='auto',
)

print('Verifying weight files…')
missing = []
for rel in WEIGHT_URLS:
    full = pathlib.Path(TALKFORGE_DIR) / rel
    if full.is_file() and full.stat().st_size > 1024:
        print(f'  ✓ {full.name}')
    else:
        print(f'  ✗ MISSING: {rel}')
        missing.append(rel)

if missing:
    print(f'\n⚠️  {len(missing)} weight file(s) missing — re-running download…')
    model.download_weights(progress_cb=print)

if model.is_ready():
    import torch
    print(f'\n✅ All weights present. Model ready!')
    print(f'   Device: {model.device}')
    print(f'   CUDA available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'   GPU: {torch.cuda.get_device_name(0)}')
else:
    raise RuntimeError('❌ Setup failed — some weights still missing. Check output above.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 9 — Launch TalkForge  🚀
# ─────────────────────────────────────────────────────────────────────────────
# A public URL (https://xxxxxxxxxxxx.gradio.live) will be printed below.
# Click it to open TalkForge in your browser.
#
# The URL stays alive for 72 hours or until this cell is stopped.
# To get a new URL: Runtime → Restart session → run all cells again.

import sys, os

# Fresh module imports so code changes take effect without restarting runtime
for mod in list(sys.modules.keys()):
    if mod.startswith(('models', 'app')):
        del sys.modules[mod]

os.chdir(TALKFORGE_DIR)
if TALKFORGE_DIR not in sys.path:
    sys.path.insert(0, TALKFORGE_DIR)

from app.interface import build_interface
from app.pipeline import init_model

WEIGHTS_DIR = f'{TALKFORGE_DIR}/weights'
OUTPUT_DIR  = f'{TALKFORGE_DIR}/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

init_model(weights_dir=WEIGHTS_DIR)

demo = build_interface(output_dir=OUTPUT_DIR, weights_dir=WEIGHTS_DIR)

print('\n' + '=' * 64)
print('  TalkForge is launching…')
print('  The public URL will appear below 👇  (may take 15–30 s)')
print('=' * 64 + '\n')

demo.queue(max_size=5).launch(
    share=True,           # creates the public gradio.live URL
    debug=False,
    show_error=True,
    server_name='0.0.0.0',
    server_port=7860,
)

---

## How to use TalkForge

1. **Upload a portrait** — any clear, front-facing photo (JPG / PNG / WEBP).
2. **Upload audio** — a speech clip (WAV / MP3 / M4A / OGG).
3. Click **✦ Generate Video** and watch the live status updates.
4. The finished video plays automatically when ready.
5. **⬇ Download Video** · **✕ Discard** · **↺ Generate New**

---

## Tips for best results

| | Details |
|---|---|
| Portrait | Clear, front-facing, well-lit. No sunglasses or heavy occlusion. Min 256×256 px. |
| Audio | 16 kHz mono WAV gives the cleanest lip sync. MP3 and M4A also work. |
| Clip length | Under 60 s for fastest generation. Longer clips work but take more time. |
| Save your videos | Colab storage is **temporary**. Download before the session ends. |

## Troubleshooting

| Problem | Fix |
|---------|-----|
| `CUDA out of memory` | Restart runtime (Runtime → Restart session) then re-run all cells |
| `Face not detected` | Use a clearer, front-facing portrait with no heavy occlusion |
| `No module named …` | Re-run Cells 5 and 6 |
| Public URL not appearing | Wait 30 s; Gradio's share server can be slow to start |
| Generation very slow | Confirm GPU runtime is enabled (Cell 1 should show a GPU name) |
| `BFM_model_front.mat` warning | Normal on first run — SadTalker generates it from source files |
| `ast.Str` / `ast.Num` error | Re-run Cell 6 to apply the basicsr Python 3.12 patch |
| Session expired | Re-run all cells — weights are cached so setup takes ~30 s |

---

## Architecture note — checkpoint sources

The original notebook used incorrect HuggingFace URLs (`vinthony/SadTalker-V0.0.2`) 
that no longer exist. The corrected sources are:

| File | Source |
|------|--------|
| `SadTalker_V0.0.2_256.safetensors` | GitHub Releases — `OpenTalker/SadTalker@v0.0.2-rc` |
| `mapping_*.pth.tar` | GitHub Releases — `OpenTalker/SadTalker@v0.0.2-rc` |
| `BFM_Fitting/*.mat` | HuggingFace — `vinthony/SadTalker` |
| `GFPGANv1.4.pth` | GitHub Releases — `TencentARC/GFPGAN@v1.3.0` |
| `alignment_WFLW_4HG.pth` | GitHub Releases — `xinntao/facexlib@v0.1.0` |
| `detection_Resnet50_Final.pth` | GitHub Releases — `xinntao/facexlib@v0.1.0` |
| `parsing_parsenet.pth` | GitHub Releases — `xinntao/facexlib@v0.2.2` |

**`BFM_model_front.mat`** is not downloaded — SadTalker generates it automatically
on first run by calling `transferBFM09()` with `01_MorphableModel.mat` + `Exp_Pca.bin`.

---

*TalkForge — MIT License.  
Powered by [SadTalker](https://github.com/OpenTalker/SadTalker) and [Gradio](https://gradio.app).*